WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import functools
from typing import Tuple

import chex
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import crops.src.bc as _bc
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.mma as _mma
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)

### Target pose generator 

In [ ]:
def cubic_target_shape(
  arc_coords: jnp.ndarray,
  quadratic_coeffs: jnp.ndarray,
  cubic_coeffs: jnp.ndarray,
) -> jnp.ndarray:
  """Generate clamped-admissible target deflection profiles, one per pose.

  The target is the transverse deflection

      y(x) = c₂ x² + c₃ x³

  evaluated at the reference arc coordinate of every node. A clamped root imposes
  both `y(0) = 0` and `y'(0) = 0`.

  Args:
    arc_coords: Array of shape `(num_nodes,)` [m]. Reference arc coordinate of each
      node, running from zero at the clamped root to the rod length at the tip.
    quadratic_coeffs: Array of shape `(num_poses,)` [1/m]. The coefficient `c₂` for
      each pose.
    cubic_coeffs: Array of shape `(num_poses,)` [1/m²]. The coefficient `c₃` for
      each pose.

  Returns:
    Array of shape `(num_poses, num_nodes)` [m] holding the target transverse
      deflection at every node for every pose.
  """
  return jnp.einsum("p, n -> pn", quadratic_coeffs, arc_coords**2) + jnp.einsum(
    "p, n -> pn", cubic_coeffs, arc_coords**3
  )


### Design variables

In [ ]:
@chex.dataclass
class DesignExtents:
  """Physical ranges for every magnetic design variable."""

  remnant_polar: _utils.Extent
  remnant_azith: _utils.Extent
  remnant_strength: _utils.Extent
  applied_polar: _utils.Extent
  applied_azith: _utils.Extent
  applied_strength: _utils.Extent

In [ ]:
def design_vars_to_opt_vars(
  design_vars: jnp.ndarray,
  extents: DesignExtents,
  num_poses: int,
  num_elems: int,
  num_fibers: int,
) -> Tuple[jnp.ndarray, jnp.ndarray]:
  """Expand the normalized MMA design vector into physical magnetic flux fields.

  The design vector is a single flat array in `[0, 1]`

      [ remanent block : 3 · num_elems · num_fibers ][ applied block : 3 · num_poses ]

  Then,
  `num_design_var = 3·num_elems·num_fibers + 3·num_poses`

  The remanent block carries `(polar, azimuthal, strength)` for each elem of each fiber.
  The applied block reshapes to `(num_poses, 3)`.

  Returns:
    A tuple `(remnant_mag_flux, applied_mag_flux)`. The first has shape
      `(num_elems, num_fibers, num_dim=3)` [T] and the second has shape
      `(num_poses, num_dim=3)` [T].
  """
  num_remnant_vars = 3 * num_elems * num_fibers
  remnant_block = design_vars[:num_remnant_vars].reshape(num_elems, num_fibers, 3)
  applied_block = design_vars[num_remnant_vars:].reshape(num_poses, 3)

  remnant_mag_flux = _utils.polar_azith_to_vector(
    extents.remnant_polar.renormalize_array(remnant_block[..., 0]),
    extents.remnant_azith.renormalize_array(remnant_block[..., 1]),
    extents.remnant_strength.renormalize_array(remnant_block[..., 2]),
  )
  applied_mag_flux = _utils.polar_azith_to_vector(
    extents.applied_polar.renormalize_array(applied_block[:, 0]),
    extents.applied_azith.renormalize_array(applied_block[:, 1]),
    extents.applied_strength.renormalize_array(applied_block[:, 2]),
  )
  return remnant_mag_flux, applied_mag_flux

In [ ]:
def opt_vars_to_design_vars(
  remnant_mag_flux: jnp.ndarray,
  applied_mag_flux: jnp.ndarray,
  extents: DesignExtents,
) -> jnp.ndarray:
  """Collapse physical magnetic flux fields into the normalized MMA design vector.

  This is the inverse of `design_vars_to_opt_vars`.

  Args:
    remnant_mag_flux: Array of shape `(num_elems, num_fibers, num_dim=3)` [T]. The
      remanent flux density of each fiber in the global reference frame.
    applied_mag_flux: Array of shape `(num_poses, num_dim=3)` [T]. The applied flux
      density for each pose in the global current frame.
    extents: `DesignExtents` supplying the physical range of each field.

  Returns:
    Array of shape `(num_design_var,)` holding the normalized design variables,
      remanent block first and applied block second.
  """
  remnant_polar, remnant_azith, remnant_strength = _utils.vector_to_polar_azith(
    remnant_mag_flux
  )
  applied_polar, applied_azith, applied_strength = _utils.vector_to_polar_azith(
    applied_mag_flux
  )

  remnant_block = jnp.stack(
    [
      extents.remnant_polar.normalize_array(remnant_polar),
      extents.remnant_azith.normalize_array(remnant_azith),
      extents.remnant_strength.normalize_array(remnant_strength),
    ],
    axis=-1,
  )
  applied_block = jnp.stack(
    [
      extents.applied_polar.normalize_array(applied_polar),
      extents.applied_azith.normalize_array(applied_azith),
      extents.applied_strength.normalize_array(applied_strength),
    ],
    axis=-1,
  )
  return jnp.concatenate([remnant_block.reshape(-1), applied_block.reshape(-1)])

### Geom

In [ ]:
rod_length = 2.0
num_elems = 100
num_nodes = num_elems + 1
fiber_offset = 0.1  # c [m]
fiber_radius = 0.02

In [ ]:
# Four fibers at (+/-c, 0) and (0, +/-c).
offsets_local = np.array(
  [[fiber_offset, 0.0], [-fiber_offset, 0.0], [0.0, fiber_offset], [0.0, -fiber_offset]]
)
num_fibers = offsets_local.shape[0]
area_per_fiber = np.pi * fiber_radius**2

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)

In [ ]:
extents = DesignExtents(
  remnant_polar=_utils.Extent(min=0.40 * np.pi, max=0.60 * np.pi),
  remnant_azith=_utils.Extent(min=-np.pi, max=np.pi),
  remnant_strength=_utils.Extent(min=0.02, max=0.40),
  applied_polar=_utils.Extent(min=0.40 * np.pi, max=0.60 * np.pi),
  applied_azith=_utils.Extent(min=-np.pi, max=np.pi),
  applied_strength=_utils.Extent(min=1.0e-4, max=4.0e-2),
)


### Target shape

In [ ]:
arc_coords = rod.arc_coords
target_shape = cubic_target_shape(
  arc_coords=arc_coords,
  quadratic_coeffs=jnp.array([-0.030]),
  cubic_coeffs=jnp.array([+0.022]),
)
num_poses = target_shape.shape[0]

fig, ax = plt.subplots(figsize=(5.5, 4))
for pose in range(num_poses):
  ax.plot(arc_coords, target_shape[pose], label=f"pose {pose}")
ax.set_xlabel("reference arc coordinate [m]")
ax.set_ylabel("target $y$ [m]")
ax.legend()
plt.show()
print(f"peak |y| / L = {float(jnp.max(jnp.abs(target_shape))) / rod_length:.3%}")

### BC

In [ ]:
num_load_steps = 10
load_fractions = jnp.linspace(1.0 / num_load_steps, 1.0, num_load_steps)

clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_load_steps)]),
    node_targets=[np.array([0]) for _ in range(num_load_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

boundary_conditions = _bc.process_boundary_conditions(rod, clamp)

### Solver

In [ ]:
nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-6,
  "abs_threshold": 1e-6,
  "lam_min": 0.01,
  "line_search_max_iter": 15,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}

linear_settings = {
  "solver": _solver.LinearSolvers.SCIPY_SPARSE,
  "rtol": 1e-12,
}

nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
  nonlinear_settings=nonlinear_settings,
  linear_settings=linear_settings,
)

problem = _fe.CosseratProblem(
  rod=rod,
  bc=boundary_conditions,
  nr_solver=nr_solver,
)

In [ ]:
shear_modulus = 5.0e5  # G [Pa]
shear_moduli = shear_modulus * jnp.ones((num_elems, num_fibers))

In [ ]:
def compute_objective(
  design_vars: jnp.ndarray,
  target_shape: jnp.ndarray,
  extents: DesignExtents,
  num_poses: int,
) -> jnp.ndarray:
  """Mean squared transverse shape error, summed over poses and normalized."""

  remnant_mag_flux, applied_mag_flux = design_vars_to_opt_vars(
    design_vars=design_vars,
    extents=extents,
    num_poses=num_poses,
    num_elems=num_elems,
    num_fibers=num_fibers,
  )

  pose_errors = []
  for pose in range(num_poses):
    applied_ramp = jnp.einsum("s, d -> sd", load_fractions, applied_mag_flux[pose])

    u_star, _ = problem.solve(
      u0=jnp.zeros(rod.num_dofs),
      shear_moduli=shear_moduli,
      remnant_mag_flux=remnant_mag_flux,
      applied_mag_flux=applied_ramp,
    )

    deflection = u_star.reshape(num_nodes, rod.num_dofs_per_node)[:, 1]
    residual = deflection - target_shape[pose]
    pose_errors.append(jnp.einsum("n, n ->", residual, residual) / num_nodes)

  normalization = jnp.mean(target_shape**2)
  return jnp.sum(jnp.stack(pose_errors)) / normalization


objective_and_grad = jax.jit(
  jax.value_and_grad(
    functools.partial(
      compute_objective,
      target_shape=target_shape,
      extents=extents,
      num_poses=num_poses,
    )
  )
)

In [ ]:
num_design_var = 3 * num_elems * num_fibers + 3 * num_poses 

init_design_vars = jnp.full((num_design_var,), 0.5).at[-1].set(0.9)

In [ ]:
mma_params = _mma.MMAParams(
  max_iter=150,
  kkt_tol=1.0e-6,
  step_tol=1.0e-6,
  move_limit=0.05,
  num_design_var=num_design_var,
  num_cons=1,
  lower_bound=np.zeros((num_design_var, 1)),
  upper_bound=np.ones((num_design_var, 1)),
)
mma_state = _mma.init_mma(
  init_design_var=np.asarray(init_design_vars).reshape(-1, 1), mma_params=mma_params
)

# MMA requires at least one constraint.
dummy_cons = np.full((1, 1), -1.0)
dummy_grad_cons = np.zeros((1, num_design_var))

history = {"objective": [], "kkt_norm": [], "change": []}

while not mma_state.is_converged and mma_state.epoch < mma_params.max_iter:
  current_design = jnp.asarray(mma_state.x).reshape(-1)
  obj_value, obj_grad = objective_and_grad(current_design)

  mma_state = _mma.update_mma(
    mma_state=mma_state,
    mma_params=mma_params,
    obj=np.array([float(obj_value)]),
    grad_obj=np.asarray(obj_grad, dtype=np.float64).reshape(-1, 1),
    cons=dummy_cons,
    grad_cons=dummy_grad_cons,
  )
  history["objective"].append(float(obj_value))
  history["kkt_norm"].append(float(mma_state.kkt_norm))
  history["change"].append(float(mma_state.change_design_var))

  if mma_state.epoch % 10 == 0:
    print(
      f"epoch {mma_state.epoch:4d}  obj {obj_value:.6e}  "
      f"kkt {mma_state.kkt_norm:.3e}  change {mma_state.change_design_var:.3e}"
    )

In [ ]:
fig, (ax_obj, ax_kkt) = plt.subplots(1, 2, figsize=(11, 4))
ax_obj.semilogy(history["objective"])
ax_obj.set_xlabel("MMA epoch")
ax_obj.set_ylabel("normalized shape error")

ax_kkt.semilogy(history["kkt_norm"], label="KKT norm")
ax_kkt.semilogy(history["change"], label="design change")
ax_kkt.set_xlabel("MMA epoch")
ax_kkt.legend()
fig.tight_layout()
plt.show()

In [ ]:
final_design_vars = jnp.asarray(mma_state.x).reshape(-1)
final_remnant, final_applied = design_vars_to_opt_vars(
  design_vars=final_design_vars,
  extents=extents,
  num_poses=num_poses,
  num_elems=num_elems,
  num_fibers=num_fibers,
)

fig, (ax_shape, ax_mag) = plt.subplots(1, 2, figsize=(12, 4.5))
for pose in range(num_poses):
  u_final, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=shear_moduli,
    remnant_mag_flux=final_remnant,
    applied_mag_flux=jnp.einsum("s, d -> sd", load_fractions, final_applied[pose]),
  )
  deflection = np.asarray(u_final).reshape(num_nodes, -1)[:, 1]
  ax_shape.plot(
    arc_coords, target_shape[pose], "k-", lw=2, label="target" if pose == 0 else None
  )
  ax_shape.plot(
    arc_coords, deflection, "o", ms=3, mfc="none", label=f"optimized pose {pose}"
  )
ax_shape.set_xlabel("reference arc coordinate [m]")
ax_shape.set_ylabel("$y$ [m]")
ax_shape.legend()

# In-plane magnetization angle of each fiber, which is what the optimizer really
# shaped. The sign changes along the rod are what produce the reversing curvature.
mag_angle = np.degrees(np.arctan2(final_remnant[:, :, 1], final_remnant[:, :, 0]))
for fiber in range(num_fibers):
  ax_mag.plot(
    np.asarray(rod.elem_centers)[:, 0],
    mag_angle[:, fiber],
    "o-",
    ms=3,
    label=f"fiber {fiber}",
  )
ax_mag.set_xlabel("$x$ [m]")
ax_mag.set_ylabel("in-plane $B_r$ angle [deg]")
ax_mag.legend(fontsize=7)
fig.tight_layout()
plt.show()

print(f"applied field per pose [T]:\n{np.asarray(final_applied)}")